# CockroachDB Vector Store


[CockroachDB](https://www.cockroachlabs.com/) v25.2+ ships a native `VECTOR(n)` column type and the [C-SPANN](https://www.cockroachlabs.com/docs/stable/vector.html) distributed approximate nearest neighbor index. This notebook walks through using `CockroachDBVectorStore` with LlamaIndex.

`llama-index-vector-stores-postgres` depends on the `pgvector` extension, which is not available in CockroachDB. The CockroachDB integration targets CRDB's native primitives directly:

| Concern | pgvector store | CockroachDB store |
| --- | --- | --- |
| Column type | `pgvector.sqlalchemy.Vector` | native `VECTOR(dim)` |
| Index DDL | `USING hnsw (... WITH m, ef_construction)` | `CREATE VECTOR INDEX ... WITH (min_partition_size, max_partition_size)` |
| Search-time knob | `SET hnsw.ef_search` | `SET vector_search_beam_size` |
| Dialect | `postgresql+psycopg2` / `+asyncpg` | `cockroachdb+psycopg2` / `cockroachdb+asyncpg` (retry-aware) |


## Setup

Requires CockroachDB v25.2+ with the vector feature enabled at the cluster level.

In [ ]:
%pip install llama-index llama-index-cockroachdb llama-index-embeddings-openai

Start a local single-node insecure cluster (development only):

```bash
cockroach start-single-node --insecure --listen-addr=localhost:26257
cockroach sql --insecure -e "SET CLUSTER SETTING feature.vector_index.enabled = true;"
```


In [ ]:
import os
os.environ.setdefault("OPENAI_API_KEY", "sk-...")


## Build the store

In [ ]:
from llama_index.vector_stores.cockroachdb import CockroachDBVectorStore

store = CockroachDBVectorStore.from_params(
    host="localhost",
    port=26257,
    user="root",
    password="",
    database="defaultdb",
    table_name="llamaindex_demo",
    embed_dim=1536,
    distance_metric="cosine",  # or "l2", "inner_product"
    cspann_kwargs={"min_partition_size": 16, "max_partition_size": 128},
    sslmode="disable",  # local insecure cluster only
)

## Index documents

`CockroachDBVectorStore` plugs into `VectorStoreIndex` the same way any other vector store does.

In [ ]:
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding

docs = [
    Document(text="C-SPANN is CockroachDB's native ANN index."),
    Document(text="HNSW is a hierarchical small-world graph used by pgvector."),
    Document(text="CockroachDB uses Raft for strongly consistent replication."),
]

index = VectorStoreIndex.from_documents(
    docs,
    storage_context=StorageContext.from_defaults(vector_store=store),
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
)

## Query

In [ ]:
query_engine = index.as_query_engine()
response = query_engine.query("How does CockroachDB do ANN search?")
print(response)

## Tune C-SPANN at query time

`vector_search_beam_size` is a per-session knob (`SET LOCAL`) controlling recall / latency. Higher beam = better recall, slightly more latency.

In [ ]:
from llama_index.core.vector_stores import VectorStoreQuery
from llama_index.embeddings.openai import OpenAIEmbedding

embed = OpenAIEmbedding(model="text-embedding-3-small")
query_emb = embed.get_text_embedding("How does CockroachDB do ANN search?")

result = store.query(
    VectorStoreQuery(query_embedding=query_emb, similarity_top_k=3),
    vector_search_beam_size=128,
)
for node, score in zip(result.nodes, result.similarities):
    print(f"{score:.4f}  {node.get_content()[:80]}")

## Metadata filters

Supports `EQ`, `NE`, `GT`, `GTE`, `LT`, `LTE`, `IN`, `NIN`, `CONTAINS`, `TEXT_MATCH`, `TEXT_MATCH_INSENSITIVE`, `IS_EMPTY`, nestable via `MetadataFilters` with AND/OR/NOT. For frequently filtered keys, declare `indexed_metadata_keys` on the store to get JSONB-extracted BTREE indices.

In [ ]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator

filters = MetadataFilters(
    filters=[MetadataFilter(key="author", value="alice", operator=FilterOperator.EQ)]
)
retriever = index.as_retriever(filters=filters, similarity_top_k=3)
nodes = retriever.retrieve("anything")
for n in nodes:
    print(n.score, n.node.get_content()[:80])

## Async

All methods have an async counterpart via the `cockroachdb+asyncpg` dialect.

In [ ]:
result = await store.aquery(
    VectorStoreQuery(query_embedding=query_emb, similarity_top_k=3),
)

## Supported query modes

| Mode | Supported | Notes |
| --- | --- | --- |
| `DEFAULT` | yes | Vector ANN through C-SPANN |
| `MMR` | yes | Client-side rerank; `mmr_threshold`, `mmr_prefetch_factor`, `mmr_prefetch_k` |
| `HYBRID` / `SPARSE` / `TEXT_SEARCH` | no | CockroachDB has no `tsvector` yet |
